In [0]:
%sql
CREATE OR REPLACE STREAMING TABLE mavenmarket.gold.fact_inventory
CLUSTER BY (restock_date_key, store_sk)
COMMENT 'Store Inventory Restocking Events Fact Table'
AS
SELECT 
    -- Primary Surrogate Key
    ABS(HASH(TRIM(i.event_id))) AS inventory_fact_sk,
    
    TRIM(i.event_id) AS event_id,
    
    -- Date Foreign Key (mapped directly to dim_date.date_sk)
    COALESCE(
        d.date_sk, 
        CAST(date_format(i.restock_date, 'yyyyMMdd') AS INT)
    ) AS restock_date_key,
    
    -- Product Foreign Key (mapped directly to dim_product.product_sk)
    COALESCE(
        p.product_sk, 
        ABS(HASH(CAST(CAST(i.product_id AS INT) AS STRING)))
    ) AS product_sk,
    
    -- Store Foreign Key (mapped directly to dim_store.store_sk)
    COALESCE(
        s.store_sk, 
        ABS(HASH(CAST(i.store_id AS STRING)))
    ) AS store_sk,
    
    COALESCE(i.restock_quantity, 0) AS restock_quantity

FROM STREAM(mavenmarket.silver.silver_inventory) i

-- Join with dim_product using its exact business key
LEFT JOIN mavenmarket.gold.dim_product p 
    ON CAST(i.product_id AS INT) = p.product_id

-- Join with dim_store using its exact business key
LEFT JOIN mavenmarket.gold.dim_store s 
    ON i.store_id = s.store_id

-- Join with dim_date using calendar date
LEFT JOIN mavenmarket.gold.dim_date d 
    ON i.restock_date = d.calendar_date

WHERE i.restock_date IS NOT NULL;